# N089 · 桥、割点与low-link

**目标：** 根据DFS子树能否绕回祖先判断脆弱连接。

**先修：** N080, N088。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 发现时间；low；父边编号；桥；根割点特例；重边扩展。

## 从问题到算法

树边的孩子若能绕路到祖先，删掉这条树边不会断开。桥用严格条件low[child]>disc[parent]；割点用low[child]≥disc[parent]，但DFS根要单独检查孩子数。多重边必须按边ID跳过父边，否则第二条平行边会被错删。

## 最小实现

**本章接口：** `find_bridges(n, edges)`、`articulation_points(n, edges)`

In [1]:
def _undirected_lowlink(n,edges):
    adj=[[] for _ in range(n)]
    for eid,(u,v) in enumerate(edges): adj[u].append((v,eid)); adj[v].append((u,eid))
    disc=[-1]*n; low=[0]*n; parent=[-1]*n; pe=[-1]*n; children=[0]*n
    bridges=[]; cuts=set(); clock=0
    for s in range(n):
        if disc[s]!=-1: continue
        disc[s]=low[s]=clock; clock+=1; stack=[(s,iter(adj[s]))]
        while stack:
            u,it=stack[-1]
            try: v,eid=next(it)
            except StopIteration:
                stack.pop(); p=parent[u]
                if p==-1:
                    if children[u]>1: cuts.add(u)
                else:
                    low[p]=min(low[p],low[u])
                    if low[u]>disc[p]: bridges.append(pe[u])
                    if parent[p]!=-1 and low[u]>=disc[p]: cuts.add(p)
                continue
            if eid==pe[u]: continue
            if disc[v]==-1:
                parent[v]=u; pe[v]=eid; children[u]+=1
                disc[v]=low[v]=clock; clock+=1; stack.append((v,iter(adj[v])))
            else: low[u]=min(low[u],disc[v])
    return bridges,cuts,disc,low

def find_bridges(n,edges):
    ids,_,_,_=_undirected_lowlink(n,edges)
    return sorted(tuple(sorted(edges[i])) for i in ids)

def articulation_points(n,edges):
    return sorted(_undirected_lowlink(n,edges)[1])

## 正确性、前提与复杂度

low值概括DFS子树经至多一条回边可触达的最早发现时间。桥的严格不等式表示子树只能经过父边离开；割点允许回到父点本身，但这在删除父点后无效，所以采用非严格不等式。根的子树间没有祖先可连，孩子数大于1才是割点。

**代价：** O(V+E)搜索与空间；对输出排序另有O(E log E+V log V)上界。支持断开图、平行边和自环。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

edges=[(0,1),(1,2),(2,0),(1,3)]; _,_,disc,low=_undirected_lowlink(4,edges)
show_table(['顶点','发现时间','low'],[(i,disc[i],low[i]) for i in range(4)])
print('桥',find_bridges(4,edges),'割点',articulation_points(4,edges))

顶点,发现时间,low
0,0,0
1,1,0
2,2,0
3,3,3


桥 [(1, 3)] 割点 [1]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from random import Random
rng=Random(89)
def components(n,edges,removed=None):
    seen=set(); count=0
    for s in range(n):
        if s==removed or s in seen: continue
        count+=1; stack=[s]; seen.add(s)
        while stack:
            u=stack.pop()
            for a,b in edges:
                if removed in (a,b): continue
                v=b if a==u else a if b==u else None
                if v is not None and v not in seen: seen.add(v); stack.append(v)
    return count
for _ in range(180):
    n=rng.randrange(1,7); edges=[(rng.randrange(n),rng.randrange(n)) for _ in range(rng.randrange(10))]
    base=components(n,edges)
    want=sorted(tuple(sorted(e)) for i,e in enumerate(edges) if components(n,edges[:i]+edges[i+1:])>base)
    assert find_bridges(n,edges)==want
    assert articulation_points(n,edges)==[u for u in range(n) if components(n,edges,u)>base]
assert find_bridges(2,[(0,1),(0,1)])==[]
assert articulation_points(2,[(0,1)])==[]
print('N089: 所有本章断言通过')

N089: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用删边/删点暴力验证。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 把无重边题扩展到多重图并按边ID处理。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1192. Critical Connections in a Network（canonical）：[官方入口](https://leetcode.com/problems/critical-connections-in-a-network/)
- 1568. Minimum Number of Days to Disconnect Island（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。